In [1]:
import tensorflow as tf
import os
import numpy as np
import keras
from keras import layers
from tensorflow import data as tf_data
import matplotlib.pyplot as plt

data_dir = "MushroomPics"
dataset = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="categorical",
    image_size=(224, 224),
    batch_size=16,
    shuffle=True
)
#dataset = dataset.apply(tf.data.experimental.ignore_errors())

#length = len(dataset)
length = tf.data.experimental.cardinality(dataset).numpy()

train_size = int(length * 0.7)
test_size = int(length * 0.15)

"""
train = dataset.take(train_size)
test = dataset.skip(train_size).take(test_size)
val = dataset.skip(train_size + test_size)
"""
train = dataset.take(train_size)
remaining = dataset.skip(train_size)
test = remaining.take(test_size)
val = remaining.skip(test_size)

print(dataset.class_names)
print(dataset)

C:\Users\leevi\anaconda3\envs\keras\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Found 6714 files belonging to 9 classes.
['Agaricus', 'Amanita', 'Boletus', 'Cortinarius', 'Entoloma', 'Hygrocybe', 'Lactarius', 'Russula', 'Suillus']
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 9), dtype=tf.float32, name=None))>


In [2]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

In [3]:
model = tf.keras.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    data_augmentation,
    
    layers.Conv2D(32, (9,9), activation='relu'),
    layers.MaxPooling2D(),
    
    layers.Conv2D(64, (9,9), activation='relu'),
    layers.MaxPooling2D(),
    
    layers.Conv2D(128, (9,9), activation='relu'),
    layers.MaxPooling2D(),
    
    layers.Conv2D(256, (9,9), activation='relu'),
    layers.MaxPooling2D(),
    
    #layers.Flatten(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    
    layers.Dense(len(dataset.class_names))
])

C:\Users\leevi\anaconda3\envs\keras\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [4]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train,
    validation_data=val,
    epochs=64
)

Epoch 1/64
294/294 ━━━━━━━━━━━━━━━━━━━━ 470s 2s/step - accuracy: 0.2136 - loss: 2.0809 - val_accuracy: 0.2196 - val_loss: 2.0855
Epoch 2/64
294/294 ━━━━━━━━━━━━━━━━━━━━ 463s 2s/step - accuracy: 0.2341 - loss: 2.0399 - val_accuracy: 0.2196 - val_loss: 2.0861
Epoch 3/64
294/294 ━━━━━━━━━━━━━━━━━━━━ 462s 2s/step - accuracy: 0.2372 - loss: 2.0347 - val_accuracy: 0.2006 - val_loss: 2.0815
Epoch 4/64
294/294 ━━━━━━━━━━━━━━━━━━━━ 461s 2s/step - accuracy: 0.2402 - loss: 2.0256 - val_accuracy: 0.2246 - val_loss: 2.0607
Epoch 5/64
294/294 ━━━━━━━━━━━━━━━━━━━━ 463s 2s/step - accuracy: 0.2396 - loss: 2.0149 - val_accuracy: 0.2156 - val_loss: 2.0553
Epoch 6/64
294/294 ━━━━━━━━━━━━━━━━━━━━ 463s 2s/step - accuracy: 0.2611 - loss: 1.9988 - val_accuracy: 0.2385 - val_loss: 2.0314
Epoch 7/64
294/294 ━━━━━━━━━━━━━━━━━━━━ 463s 2s/step - accuracy: 0.2587 - loss: 1.9974 - val_accuracy: 0.2465 - val_loss: 2.0342
Epoch 8/64
294/294 ━━━━━━━━━━━━━━━━━━━━ 465s 2s/step - accuracy: 0.2700 - loss: 1.9897 - val_accu

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs, val_acc, label='Validation accuracy')
plt.legend()
plt.title('Accuracy')
plt.show()

plt.plot(epochs, loss, label='Training loss')
plt.plot(epochs, val_loss, label='Validation loss')
plt.legend()
plt.title('Loss')
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test)
val_loss, val_acc = model.evaluate(val)
print("Test accuracy:", test_acc)
print("Validation accuracy:", val_acc)